# MQTTT Broker

This notebook documents the development process of creating the self hosted cloud core MQTT broker.

To begin, we define a minimal mosquitto.conf to setup our broker config. This will set the listening port, disable anonymous access and point to a password file.

```conf
# Disable anonymous access
allow_anonymous false

# Set password file
password_file /mosquitto/config/passwd

# Default listener
listener 1883
```

This version of the broker will be self hosted. To facilitate this we will use Docker to containerise the MQTT broker. We will use the existing `eclipse-mosquitto` image and a yml file to orchestrate the broker and its various connections to future images.

The important parts to note is the mounted port,
```yml
# Map port 1883 from the container to the host
# This allows MQTT clients to connect to localhost:1883 on the host
ports:
    - "1883:1883"
```
and the mounted volumes,
```yml
# Mount configuration files into the container
volumes:
    # Mount custom Mosquitto config file from host to container path
    - ./broker/mosquitto.conf:/mosquitto/config/mosquitto.conf

    # Mount password file for authentication
    - ./broker/passwd:/mosquitto/config/passwd
```
this allows us to point to our generated password hash and config.

Next we need to set our user name and password, we can do this through a docker command:
```bash
docker run --rm -v "$PWD/broker:/mosquitto/config" eclipse-mosquitto \
  mosquitto_passwd -b -c /mosquitto/config/passwd smartuser testpwd
```

This will create the username 'smartuser' with the password 'testpwd'.

This is the minimal needed setup to start the broker service. We can launch the broker using:
```bash
docker-compose up -d
```
and then inspect the logs to make sure that we are up
```bash
docker logs mqtt-broker
```

To be extra sure that we are running as expected, we can use 
```bash
mosquitto_sub -h localhost -t test/topic -u smartuser -P testpwd
```
in one terminal, then in a seperate terminal
```bash
mosquitto_pub -h localhost -t test/topic -m "hello from docker mqtt" -u smartuser -P testpwd
```
which will pulish a a message in the original terminal.